# 🚀 GPU-Enhanced Video Processing Suite

## Interactive Notebook with Scene Detection and Thumbnail Generation

This notebook provides an interactive interface for GPU-accelerated video processing with visual scene detection and thumbnail generation.

## 1. Setup and Dependencies

In [ ]:
# Install required packages (run once)
!pip install -q opencv-python scenedetect gradio numpy pillow matplotlib ipywidgets

In [ ]:
# Import libraries
import os
import cv2
import numpy as np
import tempfile
import zipfile
import json
import time
import subprocess
import warnings
from pathlib import Path
from typing import List, Dict, Optional, Tuple
import logging
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import display, HTML, Video, Image as IPImage
import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed

# Suppress warnings
warnings.filterwarnings('ignore')
os.environ['OPENCV_LOG_LEVEL'] = 'ERROR'

# Configure matplotlib for inline display
%matplotlib inline
plt.rcParams['figure.figsize'] = [15, 10]
plt.rcParams['figure.dpi'] = 100

print("✅ Libraries imported successfully")

## 2. GPU Detection and Status

In [ ]:
def detect_gpu_capabilities():
    """Detect all available GPU capabilities"""
    
    info = {
        'has_nvidia_gpu': False,
        'cuda_available': False,
        'opencl_available': False,
        'nvenc_available': False,
        'nvdec_available': False,
        'gpu_name': None,
        'gpu_memory': None,
        'cuda_version': None
    }
    
    # Check NVIDIA GPU
    try:
        result = subprocess.run(
            ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
            capture_output=True, text=True, timeout=2
        )
        if result.returncode == 0:
            gpu_info = result.stdout.strip().split(',')
            info['has_nvidia_gpu'] = True
            info['gpu_name'] = gpu_info[0].strip()
            info['gpu_memory'] = gpu_info[1].strip() if len(gpu_info) > 1 else None
            
            # Get CUDA version
            result = subprocess.run(['nvidia-smi'], capture_output=True, text=True, timeout=2)
            if 'CUDA Version:' in result.stdout:
                cuda_version = result.stdout.split('CUDA Version:')[1].split()[0]
                info['cuda_version'] = cuda_version
                info['cuda_available'] = True
    except:
        pass
    
    # Check OpenCL
    try:
        build_info = cv2.getBuildInformation()
        info['opencl_available'] = 'OpenCL' in build_info and 'YES' in build_info.split('OpenCL')[1][:100]
    except:
        pass
    
    # Check FFmpeg NVENC/NVDEC
    try:
        result = subprocess.run(['ffmpeg', '-encoders'], capture_output=True, text=True, timeout=2)
        info['nvenc_available'] = 'h264_nvenc' in result.stdout
        
        result = subprocess.run(['ffmpeg', '-decoders'], capture_output=True, text=True, timeout=2)
        info['nvdec_available'] = 'h264_cuvid' in result.stdout
    except:
        pass
    
    return info

# Detect GPU
gpu_info = detect_gpu_capabilities()

# Display GPU status
if gpu_info['has_nvidia_gpu']:
    display(HTML(f"""
    <div style="background: linear-gradient(135deg, #00d4ff 0%, #00a86b 100%); 
                padding: 20px; border-radius: 10px; color: white; font-size: 16px;">
        <h3>🚀 GPU Detected!</h3>
        <p><strong>GPU:</strong> {gpu_info['gpu_name']}</p>
        <p><strong>Memory:</strong> {gpu_info['gpu_memory']}</p>
        <p><strong>CUDA:</strong> {gpu_info['cuda_version'] if gpu_info['cuda_available'] else 'Not available'}</p>
        <p><strong>OpenCL:</strong> {'✅ Available' if gpu_info['opencl_available'] else '❌ Not available'}</p>
        <p><strong>NVENC:</strong> {'✅ Available' if gpu_info['nvenc_available'] else '❌ Not available'}</p>
    </div>
    """))
else:
    display(HTML("""
    <div style="background: linear-gradient(135deg, #ff6b6b 0%, #ffd93d 100%); 
                padding: 20px; border-radius: 10px; color: white;">
        <h3>⚠️ No GPU Detected</h3>
        <p>Running in CPU mode. GPU acceleration not available.</p>
    </div>
    """))

# Enable OpenCL if available
if gpu_info['opencl_available']:
    cv2.ocl.setUseOpenCL(True)
    print("✅ OpenCL acceleration enabled")

## 3. Import Video Processing Modules

In [ ]:
# Import our custom modules
import sys
sys.path.append('.')  # Add current directory to path

try:
    from video_preprocessor import VideoPreprocessor
    from video_quality_assessor import VideoQualityAssessor
    from scene_extractor_optimized import SceneExtractorOptimized
    from scene_detector_optimized import OptimizedSceneDetector
    print("✅ Custom modules imported successfully")
except ImportError as e:
    print(f"⚠️ Error importing modules: {e}")
    print("Make sure the module files are in the same directory as this notebook")

## 4. Video Selection Widget

In [ ]:
# Create file upload widget
upload_widget = widgets.FileUpload(
    accept='.mp4,.avi,.mov,.mkv',
    multiple=False,
    description='Upload Video:',
    button_style='primary'
)

# Or select from existing videos
video_dir = Path("dataset/raw_videos")
available_videos = []

if video_dir.exists():
    available_videos = list(video_dir.glob("*.mp4"))
    video_names = [v.name for v in available_videos]
else:
    video_names = []

select_widget = widgets.Dropdown(
    options=['Upload new video'] + video_names,
    value='Upload new video',
    description='Or select:',
    style={'description_width': 'initial'}
)

display(HTML("<h3>📹 Select Video for Processing</h3>"))
display(upload_widget)
display(select_widget)

# Global variable to store video path
current_video_path = None

In [ ]:
# Process video selection
def get_video_path():
    global current_video_path
    
    if select_widget.value != 'Upload new video':
        # Use selected video
        current_video_path = str(video_dir / select_widget.value)
        print(f"✅ Selected: {select_widget.value}")
    elif upload_widget.value:
        # Save uploaded video
        uploaded_file = list(upload_widget.value.values())[0]
        content = uploaded_file['content']
        filename = uploaded_file['metadata']['name']
        
        # Save to temp directory
        temp_dir = tempfile.mkdtemp()
        current_video_path = os.path.join(temp_dir, filename)
        with open(current_video_path, 'wb') as f:
            f.write(content)
        print(f"✅ Uploaded: {filename}")
    else:
        print("⚠️ Please select or upload a video")
        return None
    
    # Display video info
    if current_video_path and os.path.exists(current_video_path):
        cap = cv2.VideoCapture(current_video_path)
        if cap.isOpened():
            width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
            height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
            fps = cap.get(cv2.CAP_PROP_FPS)
            frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            duration = frames / fps if fps > 0 else 0
            size_mb = os.path.getsize(current_video_path) / (1024 * 1024)
            cap.release()
            
            display(HTML(f"""
            <div style="background: #f0f0f0; padding: 15px; border-radius: 8px; margin: 10px 0;">
                <h4>📊 Video Information</h4>
                <ul>
                    <li><strong>Resolution:</strong> {width}x{height}</li>
                    <li><strong>FPS:</strong> {fps:.2f}</li>
                    <li><strong>Duration:</strong> {duration:.2f} seconds</li>
                    <li><strong>Frames:</strong> {frames:,}</li>
                    <li><strong>Size:</strong> {size_mb:.2f} MB</li>
                </ul>
            </div>
            """))
            
            # Show video preview
            display(Video(current_video_path, width=600, height=400))
    
    return current_video_path

# Run this cell to process selection
video_path = get_video_path()

## 5. Scene Detection with Thumbnails

In [ ]:
# Scene detection parameters
detector_widget = widgets.Dropdown(
    options=['content', 'adaptive', 'threshold', 'histogram'],
    value='content',
    description='Detector:'
)

threshold_widget = widgets.IntSlider(
    value=30,
    min=10,
    max=100,
    step=5,
    description='Threshold:',
    style={'description_width': 'initial'}
)

min_scene_widget = widgets.FloatSlider(
    value=0.5,
    min=0.1,
    max=5.0,
    step=0.1,
    description='Min Scene (s):',
    style={'description_width': 'initial'}
)

use_gpu_widget = widgets.Checkbox(
    value=True,
    description='Use GPU Acceleration',
    style={'description_width': 'initial'}
)

display(HTML("<h3>⚙️ Scene Detection Settings</h3>"))
display(widgets.HBox([detector_widget, threshold_widget]))
display(widgets.HBox([min_scene_widget, use_gpu_widget]))

In [ ]:
def extract_scene_thumbnails(video_path, scenes, max_thumbs=24):
    """Extract thumbnail images for each scene"""
    thumbnails = []
    
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return thumbnails
    
    fps = cap.get(cv2.CAP_PROP_FPS)
    scenes_to_process = scenes[:max_thumbs]
    
    for i, (start_time, end_time) in enumerate(scenes_to_process):
        # Get middle frame of the scene
        middle_time = (start_time.get_seconds() + end_time.get_seconds()) / 2
        middle_frame = int(middle_time * fps)
        
        cap.set(cv2.CAP_PROP_POS_FRAMES, middle_frame)
        ret, frame = cap.read()
        
        if ret:
            # Resize for thumbnail
            height, width = frame.shape[:2]
            thumb_width = 320
            thumb_height = int(height * (thumb_width / width))
            thumbnail = cv2.resize(frame, (thumb_width, thumb_height))
            
            # Convert BGR to RGB
            thumbnail_rgb = cv2.cvtColor(thumbnail, cv2.COLOR_BGR2RGB)
            
            # Add text overlay
            duration = end_time.get_seconds() - start_time.get_seconds()
            text = f"Scene {i+1} ({duration:.1f}s)"
            cv2.putText(thumbnail_rgb, text, (10, 30), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv2.putText(thumbnail_rgb, f"{start_time.get_seconds():.1f}s - {end_time.get_seconds():.1f}s", 
                       (10, thumb_height-10), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
            
            thumbnails.append(thumbnail_rgb)
    
    cap.release()
    return thumbnails

# Global variables for scenes
detected_scenes = []
scene_thumbnails = []

In [ ]:
# Detect scenes
def detect_scenes():
    global detected_scenes, scene_thumbnails
    
    if not current_video_path:
        print("⚠️ Please select a video first")
        return
    
    # Enable GPU if requested
    if use_gpu_widget.value and gpu_info['opencl_available']:
        cv2.ocl.setUseOpenCL(True)
        print("✅ GPU acceleration enabled")
    
    # Create detector
    detector = OptimizedSceneDetector(
        detector_type=detector_widget.value,
        threshold=threshold_widget.value,
        min_scene_len=min_scene_widget.value,
        use_multicore=True
    )
    
    print(f"🔍 Detecting scenes with {detector_widget.value} detector...")
    start_time = time.time()
    
    # Detect scenes
    detected_scenes = detector.detect_scenes(current_video_path, show_progress=False)
    detect_time = time.time() - start_time
    
    print(f"✅ Found {len(detected_scenes)} scenes in {detect_time:.2f} seconds")
    
    if detected_scenes:
        # Extract thumbnails
        print("🖼️ Generating thumbnails...")
        scene_thumbnails = extract_scene_thumbnails(current_video_path, detected_scenes)
        print(f"✅ Generated {len(scene_thumbnails)} thumbnails")
        
        # Display thumbnails in grid
        if scene_thumbnails:
            cols = 4
            rows = (len(scene_thumbnails) + cols - 1) // cols
            
            fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 4))
            axes = axes.flatten() if rows > 1 else [axes] if rows == 1 and cols == 1 else axes
            
            for i, thumb in enumerate(scene_thumbnails):
                axes[i].imshow(thumb)
                axes[i].axis('off')
            
            # Hide empty subplots
            for i in range(len(scene_thumbnails), len(axes)):
                axes[i].axis('off')
            
            plt.suptitle(f'Detected {len(detected_scenes)} Scenes', fontsize=16, fontweight='bold')
            plt.tight_layout()
            plt.show()
        
        # Display scene statistics
        durations = [(end.get_seconds() - start.get_seconds()) for start, end in detected_scenes]
        
        display(HTML(f"""
        <div style="background: #e8f4f8; padding: 15px; border-radius: 8px; margin: 10px 0;">
            <h4>📊 Scene Statistics</h4>
            <ul>
                <li><strong>Total Scenes:</strong> {len(detected_scenes)}</li>
                <li><strong>Average Duration:</strong> {np.mean(durations):.2f} seconds</li>
                <li><strong>Min Duration:</strong> {np.min(durations):.2f} seconds</li>
                <li><strong>Max Duration:</strong> {np.max(durations):.2f} seconds</li>
                <li><strong>Detection Time:</strong> {detect_time:.2f} seconds</li>
                <li><strong>GPU Used:</strong> {'✅ Yes' if use_gpu_widget.value and gpu_info['opencl_available'] else '❌ No'}</li>
            </ul>
        </div>
        """))
    else:
        print("⚠️ No scenes detected")

# Run detection
detect_button = widgets.Button(
    description='🔍 Detect Scenes',
    button_style='success',
    layout=widgets.Layout(width='200px', height='40px')
)

detect_button.on_click(lambda b: detect_scenes())
display(detect_button)

## 6. Extract and Save Scenes

In [ ]:
def extract_scenes():
    if not detected_scenes:
        print("⚠️ No scenes to extract. Run scene detection first.")
        return
    
    print(f"✂️ Extracting {len(detected_scenes)} scenes...")
    
    # Create output directory
    temp_dir = tempfile.mkdtemp()
    scenes_dir = os.path.join(temp_dir, "scenes")
    
    # Extract scenes
    extractor = SceneExtractorOptimized(
        threshold=threshold_widget.value,
        min_scene_len=int(min_scene_widget.value * 30),  # Convert to frames
        use_multicore=True
    )
    
    start_time = time.time()
    extracted = extractor.extract_scenes(current_video_path, scenes_dir)
    extract_time = time.time() - start_time
    
    if extracted:
        print(f"✅ Extracted {len(extracted)} scenes in {extract_time:.2f} seconds")
        
        # Create ZIP package
        zip_path = os.path.join(temp_dir, "scenes_package.zip")
        with zipfile.ZipFile(zip_path, 'w') as zipf:
            # Add scene videos
            for scene in extracted:
                if os.path.exists(scene['output_path']):
                    arcname = os.path.basename(scene['output_path'])
                    zipf.write(scene['output_path'], arcname)
            
            # Add thumbnails
            if scene_thumbnails:
                for i, thumb in enumerate(scene_thumbnails):
                    thumb_path = os.path.join(temp_dir, f"thumb_{i:03d}.jpg")
                    Image.fromarray(thumb).save(thumb_path, "JPEG")
                    zipf.write(thumb_path, f"thumbnails/thumb_{i:03d}.jpg")
            
            # Add metadata
            metadata = {
                'total_scenes': len(extracted),
                'extraction_time': extract_time,
                'gpu_used': gpu_info['opencl_available'],
                'scenes': extracted
            }
            zipf.writestr('metadata.json', json.dumps(metadata, indent=2))
        
        file_size = os.path.getsize(zip_path) / (1024 * 1024)
        
        display(HTML(f"""
        <div style="background: #d4f4dd; padding: 15px; border-radius: 8px; margin: 10px 0;">
            <h4>✅ Extraction Complete!</h4>
            <p><strong>Package:</strong> scenes_package.zip</p>
            <p><strong>Size:</strong> {file_size:.2f} MB</p>
            <p><strong>Contents:</strong></p>
            <ul>
                <li>{len(extracted)} scene videos</li>
                <li>{len(scene_thumbnails)} thumbnails</li>
                <li>metadata.json</li>
            </ul>
            <p><strong>Location:</strong> <code>{zip_path}</code></p>
        </div>
        """))
        
        # Create download link
        from IPython.display import FileLink
        display(FileLink(zip_path, result_html_prefix="📦 Download: "))
    else:
        print("❌ Failed to extract scenes")

# Extract button
extract_button = widgets.Button(
    description='✂️ Extract Scenes',
    button_style='primary',
    layout=widgets.Layout(width='200px', height='40px')
)

extract_button.on_click(lambda b: extract_scenes())
display(extract_button)

## 7. GPU Performance Benchmark

In [ ]:
def benchmark_gpu():
    if not current_video_path:
        print("⚠️ Please select a video first")
        return
    
    print("⚡ Running GPU benchmark...")
    
    # Test with OpenCL enabled vs disabled
    results = {}
    
    if gpu_info['opencl_available']:
        # Test with OpenCL
        cv2.ocl.setUseOpenCL(True)
        start = time.time()
        
        cap = cv2.VideoCapture(current_video_path)
        frames_processed = 0
        while frames_processed < 100 and cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            # Perform GPU operations
            gpu_frame = cv2.UMat(frame)
            resized = cv2.resize(gpu_frame, (256, 256))
            gray = cv2.cvtColor(resized, cv2.COLOR_BGR2GRAY)
            frames_processed += 1
        cap.release()
        
        results['gpu_time'] = time.time() - start
        
        # Test without OpenCL
        cv2.ocl.setUseOpenCL(False)
        start = time.time()
        
        cap = cv2.VideoCapture(current_video_path)
        frames_processed = 0
        while frames_processed < 100 and cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            # CPU operations
            resized = cv2.resize(frame, (256, 256))
            gray = cv2.cvtColor(resized, cv2.COLOR_BGR2GRAY)
            frames_processed += 1
        cap.release()
        
        results['cpu_time'] = time.time() - start
        results['speedup'] = results['cpu_time'] / results['gpu_time'] if results['gpu_time'] > 0 else 1.0
        
        # Re-enable OpenCL
        cv2.ocl.setUseOpenCL(True)
        
        # Display results
        display(HTML(f"""
        <div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); 
                    padding: 20px; border-radius: 10px; color: white;">
            <h3>⚡ Benchmark Results</h3>
            <table style="width: 100%; color: white;">
                <tr>
                    <td><strong>CPU Time:</strong></td>
                    <td>{results['cpu_time']:.3f} seconds</td>
                </tr>
                <tr>
                    <td><strong>GPU Time (OpenCL):</strong></td>
                    <td>{results['gpu_time']:.3f} seconds</td>
                </tr>
                <tr>
                    <td><strong>Speedup:</strong></td>
                    <td>{results['speedup']:.2f}x faster</td>
                </tr>
            </table>
        </div>
        """))
        
        # Plot comparison
        fig, ax = plt.subplots(1, 1, figsize=(8, 6))
        categories = ['CPU', 'GPU (OpenCL)']
        times = [results['cpu_time'], results['gpu_time']]
        colors = ['#ff6b6b', '#51cf66']
        
        bars = ax.bar(categories, times, color=colors)
        ax.set_ylabel('Time (seconds)', fontsize=12)
        ax.set_title('CPU vs GPU Performance', fontsize=14, fontweight='bold')
        
        # Add value labels on bars
        for bar, time in zip(bars, times):
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{time:.3f}s', ha='center', va='bottom')
        
        plt.tight_layout()
        plt.show()
    else:
        print("❌ No GPU acceleration available for benchmarking")

# Benchmark button
benchmark_button = widgets.Button(
    description='⚡ Run Benchmark',
    button_style='warning',
    layout=widgets.Layout(width='200px', height='40px')
)

benchmark_button.on_click(lambda b: benchmark_gpu())
display(benchmark_button)

## 8. Interactive Scene Viewer

In [ ]:
# Interactive scene viewer
def view_scene(scene_index):
    if not detected_scenes:
        print("No scenes detected yet")
        return
    
    if scene_index < 0 or scene_index >= len(detected_scenes):
        print("Invalid scene index")
        return
    
    start_time, end_time = detected_scenes[scene_index]
    duration = end_time.get_seconds() - start_time.get_seconds()
    
    # Display scene info
    display(HTML(f"""
    <div style="background: #f8f9fa; padding: 15px; border-radius: 8px; margin: 10px 0;">
        <h4>Scene {scene_index + 1}</h4>
        <ul>
            <li><strong>Start:</strong> {start_time.get_seconds():.2f} seconds</li>
            <li><strong>End:</strong> {end_time.get_seconds():.2f} seconds</li>
            <li><strong>Duration:</strong> {duration:.2f} seconds</li>
        </ul>
    </div>
    """))
    
    # Display thumbnail if available
    if scene_index < len(scene_thumbnails):
        plt.figure(figsize=(8, 6))
        plt.imshow(scene_thumbnails[scene_index])
        plt.axis('off')
        plt.title(f'Scene {scene_index + 1} Preview')
        plt.show()

# Create interactive slider
if detected_scenes:
    scene_slider = widgets.IntSlider(
        value=0,
        min=0,
        max=len(detected_scenes)-1,
        step=1,
        description='Scene:',
        continuous_update=False
    )
    
    display(HTML("<h3>👁️ Interactive Scene Viewer</h3>"))
    interact(view_scene, scene_index=scene_slider)
else:
    print("⚠️ No scenes detected yet. Run scene detection first.")

## 9. Batch Processing

In [ ]:
# Batch process multiple videos
def batch_process_videos():
    if not video_dir.exists():
        print("⚠️ No video directory found")
        return
    
    videos = list(video_dir.glob("*.mp4"))
    if not videos:
        print("⚠️ No videos found in directory")
        return
    
    print(f"📁 Found {len(videos)} videos to process")
    
    results = []
    
    for video in videos:
        print(f"\n🎬 Processing: {video.name}")
        
        try:
            # Detect scenes
            detector = OptimizedSceneDetector(
                detector_type='content',
                threshold=30,
                min_scene_len=0.5,
                use_multicore=True
            )
            
            scenes = detector.detect_scenes(str(video), show_progress=False)
            
            results.append({
                'video': video.name,
                'scenes': len(scenes),
                'status': '✅'
            })
            
            print(f"  ✅ Found {len(scenes)} scenes")
            
        except Exception as e:
            results.append({
                'video': video.name,
                'scenes': 0,
                'status': '❌'
            })
            print(f"  ❌ Error: {e}")
    
    # Display results table
    display(HTML("<h4>📊 Batch Processing Results</h4>"))
    
    import pandas as pd
    df = pd.DataFrame(results)
    display(df)

# Batch process button
batch_button = widgets.Button(
    description='📁 Batch Process',
    button_style='info',
    layout=widgets.Layout(width='200px', height='40px')
)

batch_button.on_click(lambda b: batch_process_videos())
display(batch_button)

## 10. Summary and Export

In [ ]:
# Generate processing summary
def generate_summary():
    summary = {
        'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
        'gpu_available': gpu_info['has_nvidia_gpu'],
        'gpu_name': gpu_info.get('gpu_name', 'N/A'),
        'cuda_version': gpu_info.get('cuda_version', 'N/A'),
        'opencl_enabled': gpu_info['opencl_available'],
        'video_processed': current_video_path if current_video_path else 'None',
        'scenes_detected': len(detected_scenes) if detected_scenes else 0,
        'thumbnails_generated': len(scene_thumbnails) if scene_thumbnails else 0
    }
    
    display(HTML(f"""
    <div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); 
                padding: 20px; border-radius: 10px; color: white;">
        <h3>📊 Processing Summary</h3>
        <p><strong>Timestamp:</strong> {summary['timestamp']}</p>
        <p><strong>GPU:</strong> {summary['gpu_name']}</p>
        <p><strong>CUDA:</strong> {summary['cuda_version']}</p>
        <p><strong>OpenCL:</strong> {'✅ Enabled' if summary['opencl_enabled'] else '❌ Disabled'}</p>
        <p><strong>Scenes Detected:</strong> {summary['scenes_detected']}</p>
        <p><strong>Thumbnails Generated:</strong> {summary['thumbnails_generated']}</p>
    </div>
    """))
    
    # Save summary to JSON
    summary_path = 'processing_summary.json'
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=2)
    
    print(f"\n📄 Summary saved to: {summary_path}")

# Generate summary button
summary_button = widgets.Button(
    description='📊 Generate Summary',
    button_style='success',
    layout=widgets.Layout(width='200px', height='40px')
)

summary_button.on_click(lambda b: generate_summary())
display(summary_button)

---

## 🎉 Notebook Complete!

This notebook provides a complete GPU-enhanced video processing pipeline with:
- Scene detection with multiple algorithms
- Automatic thumbnail generation
- GPU acceleration via OpenCL and NVENC
- Interactive scene viewing
- Batch processing capabilities

**GPU Status:** Check cell 2 for your GPU configuration

**Next Steps:**
1. Upload or select a video
2. Configure scene detection parameters
3. Run scene detection to see thumbnails
4. Extract scenes for download
5. Benchmark GPU performance

---